# Arc — CUDA build check + QTIP parity dry-run on a free/cheap GPU (Google Colab)

**Open in Colab (one click):**
<https://colab.research.google.com/github/aeonmindai/arc/blob/master/arc-tools/colab_cuda_build_check.ipynb>

(Or: colab.research.google.com → File → Open notebook → GitHub tab → `aeonmindai/arc` → this path. The repo is public, so no GitHub auth is needed.)

**What this does:** clones Arc at a chosen commit and compiles every Arc-authored
CUDA kernel (QTIP + arc-cuda-graph) using Colab's free `nvcc`. Compiling CUDA needs
only the toolkit, *not* a matching GPU — so we cross-compile for **sm_90 (H100/Hopper)**,
the V4 Flash rental target, regardless of which GPU Colab hands you.

**Two tiers, depending on the GPU Colab gives you:**

| GPU | Cost | What you get |
|---|---|---|
| **T4 (sm_75)** — free | $0 | Compile gate only (kernels build for sm_90). QTIP can't *run* (gated sm_80+). |
| **A100 (sm_80)** — Colab Pro (~$10/mo) | ~cents/run | **Faithful dry-run of the rental's step-4b pre-download gate**: the exact 7 QTIP parity tests `rental_h100_v4_flash.sh` runs *before* the ~148 GB V4 download. |

The A100 tier is the point: the single most dangerous M1 gate is QTIP kernel
**runtime** parity (Viterbi quantize / fused gemv / rotation), which no Mac and no
free T4 can execute. A Colab Pro A100 clears it for ~cents instead of discovering a
broken kernel after paying for an H100 + a 148 GB pull. **To get an A100:**
`Runtime → Change runtime type → A100 GPU` (requires Colab Pro).

**Honest scope / limits:**
- The compile gate is identical to `.github/workflows/cuda_compile_check.yaml` and
  `arc-tools/cuda_compile_check.sh`. It proves the kernels **compile** for the rental arch.
- This does **not** run flash-attn (kept off for Colab RAM/disk) and does **not** do the
  final CLI `-lcuda` link — those stay on the rental. It also does **not** load real V4
  weights. It de-risks the *kernel-parity* gate, not the full M1 gate (see `M1_GATE.md`).
- Use **GitHub Actions** (free, automatic, no GPU) as the primary compile gate; this
  notebook is the second free path that uses a real `nvcc` and, on A100, real kernels.

Set the runtime to a GPU before running (`Runtime → Change runtime type → GPU`).

## 1. Inspect the GPU + CUDA toolkit Colab gave you

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv || echo 'no GPU — compile-only still works'
!nvcc --version

## 2. Choose the commit to validate
Defaults to `master`. Pin to a SHA to validate exactly what the rental will check out.

In [ ]:
ARC_REPO = 'https://github.com/aeonmindai/arc.git'
ARC_REF  = 'master'   # e.g. '12527af2d' to pin an exact commit
print('repo:', ARC_REPO, '\nref :', ARC_REF)

## 3. Install Rust (stable)

In [ ]:
import os
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain stable --profile minimal
os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']
!cargo --version

## 4. Clone Arc at the chosen ref

In [ ]:
import os
!rm -rf /content/arc
!git clone --filter=blob:none $ARC_REPO /content/arc
!cd /content/arc && git checkout $ARC_REF && git log -1 --oneline
os.chdir('/content/arc')

## 5. Run the CUDA compile gate (sm_90)
Uses `arc-tools/cuda_compile_check.sh`. `FEATURES=cuda` (no flash-attn) keeps the build
within Colab's RAM/disk. `CUDA_COMPUTE_CAP=90` cross-compiles the QTIP kernels for Hopper
even on a T4. Because the compile target is forced to sm_90, the script's step-5 runtime
smoke tests run **only on a matching sm_90 GPU** (the H100 rental); on a T4 or A100 it
reports `SKIP` and stays compile-only — you can't run sm_90 binaries on an older GPU
(doing so fails with `CUDA_ERROR_INVALID_PTX`). Runtime QTIP parity on an A100 is exercised
by the **next cell**, which compiles for your GPU's actual arch.

In [ ]:
!cd /content/arc && \
  CUDA_COMPUTE_CAP=90 FEATURES=cuda RUN_GPU_TESTS=auto \
  bash arc-tools/cuda_compile_check.sh

## 6. QTIP GPU parity — the rental step-4b dry-run (A100 / sm_80+ only)

Runs the **exact 7 parity tests** that `arc-tools/rental_h100_v4_flash.sh` step 4b
runs on the H100 *before* the 148 GB download — the gate that catches a broken/hung
QTIP kernel in ~1 min instead of after a long pull + 30 min of ISQ. On a T4 (sm_75)
the QTIP kernels aren't compiled in, so this cell reports that and stops (expected,
not an Arc failure). The cell mirrors the script's **anti-skip guards**: the tests
print `"CUDA not available; skipping"` and still return `ok` if the CUDA device
isn't usable, so we assert the real `cos sim` parity output is present and reject a
silent skip — otherwise a green "test result: ok" could be a no-op.

In [ ]:
# The exact 7 QTIP parity tests from rental_h100_v4_flash.sh step 4b, with the
# script's anti-skip guards. Runs only on sm_80+ (A100/H100); no-op-reports on T4.
import subprocess, sys

cc = subprocess.run(['bash','-lc',
   "nvidia-smi --query-gpu=compute_cap --format=csv,noheader 2>/dev/null | head -1 | tr -d '. '"],
   capture_output=True, text=True).stdout.strip() or '0'
print('compute_cap =', cc)

QTIP_TESTS = [
    'cuda_quantize_matches_cpu_dequantize_cos_sim',
    'cuda_dequantize_matches_cpu_viterbi_rotation',
    'cuda_forward_matches_cpu_viterbi_rotation',
    'cuda_rotate_x_matches_cpu',
    'cuda_fused_gemv_matches_dequant_matmul',
    'cuda_fused_gemv_matches_dequant_matmul_no_rotation',
    'qtip_gather_forward_cuda_matches_cpu',
]

if not (cc.isdigit() and int(cc) >= 80):
    print('GPU is sm_%s (< sm_80) — QTIP kernels are not enabled here.' % cc)
    print('The compile gate above already proved they build for sm_90 (the rental arch).')
    print('Runtime parity needs an A100 (Colab Pro) or the rental H100. Stopping (expected).')
else:
    cmd = (
        'cd /content/arc && CUDA_COMPUTE_CAP=%s '
        'cargo test --release -p mistralrs-quant --features cuda -- '
        '--nocapture --test-threads=1 %s'
    ) % (cc, ' '.join(QTIP_TESTS))
    print('+', cmd, '\n')
    res = subprocess.run(['bash','-lc', cmd], capture_output=True, text=True)
    log = res.stdout + res.stderr
    print(log[-6000:])  # tail
    # Same guards as the rental script: a green "ok" with no kernel run is a trap.
    failed = (res.returncode != 0)
    if 'CUDA not available; skipping' in log:
        print('\nFAIL: tests skipped — CUDA device not usable on this Colab runtime.'); failed = True
    if 'cos sim' not in log:
        print('\nFAIL: no "cos sim" parity output — kernels did not actually run.'); failed = True
    if failed:
        print('\n❌ QTIP parity dry-run FAILED — this is a code defect; fix before the rental.')
        sys.exit(1)
    print('\n✅ QTIP parity dry-run PASSED on sm_%s — rental step-4b gate cleared off the H100.' % cc)

## 7. Numerical stack-composition proxy (CPU — criterion-3 offline gate)
Optional, runs anywhere. The same RUN-151 proxy `preflight.sh` runs. Read the scope
note in the cell: it's a synthetic, deliberately-weaker proxy, **not** the real-V4
criterion-3 bar.

In [ ]:
# Criterion-3 offline proxy (CPU — runs on any runtime, even no-GPU).
# Same test preflight.sh runs. Synthetic, weaker bar than the real-V4 gate
# (see M1_GATE.md criterion 3): per-layer cos-sim >= 0.85, logits >= 0.80,
# 20 teacher-forced decode steps >= 0.75. Proves the QTIP+TurboQuant+TD-MoE
# stack composes without NaN/drift; does NOT pre-clear the 0.95/first-100-greedy
# bar — that needs real V4 weights on the rental.
!cd /content/arc && cargo test -p arc-engine --test numerical_stack_composition \
  arc_compression_stack_composes_within_drift_budget -- --nocapture 2>&1 | \
  grep -iE "cos.?sim|RUN-151|test result|layer [0-9]|final logits|PASS|FAIL"